# Activities Classifier - Pre-workshop (Overture + reviewed lookup)

**This notebook is the background work for workshop preparation**

It produces the classification lookup used by `03-NA02-POIs_Gehl.ipynb`.
Workshop participants don't need to run it: the finished lookup is part of the workshop
data in `data/reference/`.

**Requires:**
- `data/overture/places_all_cities_backup.geojson`: the places from the old Overture release (read only)
- `data/reference/BackUp/overture_place_classification_REVIEWED_old_taxonomy.csv`: the lookup
  reviewed by hand under the old category names (read only)
- Internet access and time: about 770,000 places for five cities are downloaded from Overture,
  and Section 6 downloads a ~1.6 GB language model the first time new categories appear

| Output | Path | What it is |
|---|---|---|
| Places backup | `data/overture/places_all_cities_{release}.geojson` | Overture `place` features for five Vietnamese cities, one file |
| Crosswalk | `data/reference/overture_category_crosswalk.csv` | Old Overture category → new category, with how each link was found |
| Classification lookup | `data/reference/overture_place_classification.csv` / `.json` | Every category → functional group + Gehl activity |

### Why this notebook changed (Overture release 2026-09-23)

Overture removed the `categories` column from places. The category now lives in
`taxonomy.primary`, with its full path in `taxonomy.hierarchy`
(e.g. `shopping > food_and_beverage_store > health_food_store`). About 20% of the category
names changed (`spas` → `spa`, `karaoke` → `karaoke_venue`, …).

The lookup was reviewed by hand under the **old** names
(`data/reference/BackUp/overture_place_classification_REVIEWED_old_taxonomy.csv`). This
notebook carries that review over to the new names instead of starting again:

1. Download the five cities from the new release.
2. Match every place to the old backup (`places_all_cities_backup.geojson`) by its Overture
   ID, which stays the same across releases. This shows which old category each new
   category came from.
3. Copy the reviewed columns across, by category name — never by row position.
4. Only categories with no reviewed source get a BART Gehl label and are flagged for review.

**Structure**
1. Download Overture places (new release) for five Vietnamese cities
2. Combine and save them as one backup
3. Category inventory, functional group from each place's own hierarchy
4. Crosswalk old → new categories
5. Carry the reviewed labels over
6. BART Gehl activity, only for categories without a reviewed source
7. Export the CSV
8. Rebuild the JSON from the (hand-edited) CSV

---
## Environment Setup

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Running in:", "Google Colab" if IN_COLAB else "Local environment")

Running in: Local environment


In [2]:
# Colab only: mount Google Drive
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/gdrive")

In [3]:
#@title <font color=#1B7192> Configuration - Click to Run </font>  { display-mode: "form" }
if IN_COLAB:
    print("Installing libraries...")
    %pip install -q osmnx "sitex[data] @ git+https://github.com/ArchiColab/sitex.git"
    print("Libraries installed!")

In [4]:
# Core imports
from pathlib import Path
import os
import sys
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox

from sitex.data import overture as ovt
from sitex.arch.poi_classification import (
    THIRDSPACE_EXCLUDED_GROUPS, export_classification_json, functional_group_from_hierarchy,
)

print(f"   pandas    : {pd.__version__}")
print(f"   geopandas : {gpd.__version__}")
print(f"   osmnx     : {ox.__version__}")
print("Imports OK")

   pandas    : 2.3.3
   geopandas : 1.1.3
   osmnx     : 2.0.7
Imports OK


---
# 1. Download Overture places (new release) for five Vietnamese cities

In [5]:
#@title <font color=#1B7192> Folder Path </font>  { display-mode: "form" }

if IN_COLAB:
    DATA_DIR = Path("/content/gdrive/MyDrive/Colab_Outputs")
else:
    DATA_DIR = Path("..") / "data"

OVT_DIR = DATA_DIR / "overture"
REF_DIR = DATA_DIR / "reference"
OVT_DIR.mkdir(parents=True, exist_ok=True)
REF_DIR.mkdir(parents=True, exist_ok=True)

RELEASE = ovt.get_latest_release()
print(f"Overture release: {RELEASE}")

# New-release places (written by this notebook)
NEW_BACKUP_PATH = OVT_DIR / f"places_all_cities_{RELEASE}.geojson"

# Old-release places (categories.primary) -- read only, the bridge to the reviewed CSV
OLD_BACKUP_PATH = OVT_DIR / "places_all_cities_backup.geojson"

# The hand-reviewed lookup under the OLD category names -- read only, never overwritten
REVIEWED_OLD_CSV = REF_DIR / "BackUp" / "overture_place_classification_REVIEWED_old_taxonomy.csv"

TAXONOMY_PATH = REF_DIR / "overture_taxonomy.csv"   # Overture's old -> new name sheet (cached)
CROSSWALK_CSV = REF_DIR / "overture_category_crosswalk.csv"
CLASSIFICATION_CSV = REF_DIR / "overture_place_classification.csv"
CLASSIFICATION_JSON = REF_DIR / "overture_place_classification.json"

# Overture's public "Place Categories + Basic Categories" sheet ("Latest Taxonomy + Basic"
# tab, linked from https://docs.overturemaps.org/guides/places/): old -> new category names.
# Downloaded once and cached; delete the CSV to fetch a fresh copy.
TAXONOMY_URL = (
    "https://docs.google.com/spreadsheets/d/1_i2S48zTDoHff0uX-d8Nes3bR-Xee8drx27Gyi80CQ0"
    "/export?format=csv&gid=1609295381"
)
if TAXONOMY_PATH.exists():
    print(f"  cache hit  -> {TAXONOMY_PATH}")
else:
    print("  downloading Overture's category taxonomy sheet ...")
    pd.read_csv(TAXONOMY_URL).to_csv(TAXONOMY_PATH, index=False, encoding="utf-8-sig")
    print(f"  saved -> {TAXONOMY_PATH}")

for p in (OLD_BACKUP_PATH, REVIEWED_OLD_CSV):
    if not p.exists():
        raise FileNotFoundError(f"{p} is needed to carry the reviewed labels over.")
print("Inputs found.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Overture release: 2026-09-23.1
  cache hit  -> ..\data\reference\overture_taxonomy.csv
Inputs found.


In [6]:
# Keys are filename-safe slugs; values are OSM/Nominatim place queries.
CITIES = {
    "pleiku":    "Phường Pleiku, Gia Lai, Vietnam",
    "can_tho":   "Can Tho, Vietnam",
    "da_nang":   "Da Nang, Vietnam",
    "hanoi":     "Hanoi, Vietnam",
    "hcmc":      "Ho Chi Minh City, Vietnam",
}

In [7]:
def get_city_boundary(place_query: str):
    """Geocode a place name to its OSM boundary polygon + bbox (lon/lat, EPSG:4326)."""
    city = ox.geocoder.geocode_to_gdf(place_query).to_crs("EPSG:4326")
    return city.geometry.iloc[0], tuple(city.total_bounds)


def fetch_overture_places(bbox: tuple, cache_path: Path, release: str) -> gpd.GeoDataFrame:
    """Overture places for a bbox from the given release, cached as GeoJSON.

    Since release 2026-09-23 the category is `taxonomy.primary` (the old `categories`
    column is gone); `hierarchy` keeps the full path for the functional-group mapping.
    """
    if cache_path.exists():
        print(f"  cache hit  -> {cache_path}")
        return gpd.read_file(cache_path)

    print(f"  downloading Overture places ({release}) for bbox {tuple(round(v, 4) for v in bbox)} ...")
    xmin, ymin, xmax, ymax = bbox
    s3 = f"s3://overturemaps-us-west-2/release/{release}/theme=places/type=place/*"
    con = ovt.init_duckdb()
    df = con.execute(f"""
        SELECT id, names."primary" AS names, taxonomy."primary" AS primary_category,
               array_to_string(taxonomy.hierarchy, ' > ') AS hierarchy,
               basic_category, confidence, ST_AsWKB(geometry) AS geom_wkb
        FROM read_parquet('{s3}', hive_partitioning=1)
        WHERE bbox.xmin <= {xmax} AND bbox.xmax >= {xmin}
          AND bbox.ymin <= {ymax} AND bbox.ymax >= {ymin}
    """).df()
    con.close()

    geometry = gpd.GeoSeries.from_wkb(df.pop("geom_wkb").apply(bytes), crs="EPSG:4326")
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
    gdf.to_file(cache_path, driver="GeoJSON")
    print(f"  saved {len(gdf):,} features -> {cache_path}")
    return gdf

In [8]:
city_places = {}   # slug -> GeoDataFrame clipped to the city boundary

for slug, query in CITIES.items():
    print(f"=== {slug} ({query}) ===")
    try:
        polygon, bbox = get_city_boundary(query)
    except Exception as exc:
        print(f"  ! could not geocode '{query}': {exc}\n  skipping this city.")
        continue

    gdf = fetch_overture_places(bbox, OVT_DIR / f"{slug}_places_{RELEASE}.geojson", RELEASE)
    gdf_clip = gdf[gdf.geometry.within(polygon)].reset_index(drop=True)
    gdf_clip["city"] = slug

    # Same de-duplication as the old backup: drop repeated names (NaN names are kept).
    dup_mask = gdf_clip["names"].notna() & gdf_clip["names"].duplicated(keep="first")
    if dup_mask.any():
        print(f"  dropping {dup_mask.sum():,} places with a duplicate name")
        gdf_clip = gdf_clip[~dup_mask].reset_index(drop=True)

    city_places[slug] = gdf_clip
    print(f"  {len(gdf_clip):,} places within {slug} boundary\n")

print("Cities loaded:", list(city_places.keys()))

=== pleiku (Phường Pleiku, Gia Lai, Vietnam) ===
  downloading Overture places (2026-09-23.1) for bbox (107.9951, 13.9656, 108.0634, 14.0336) ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  saved 5,439 features -> ..\data\overture\pleiku_places_2026-09-23.1.geojson
  dropping 27 places with a duplicate name
  3,535 places within pleiku boundary

=== can_tho (Can Tho, Vietnam) ===
  downloading Overture places (2026-09-23.1) for bbox (105.2257, 8.7025, 106.4538, 10.3252) ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  saved 60,400 features -> ..\data\overture\can_tho_places_2026-09-23.1.geojson


...

  dropping 14,382 places with a duplicate name
  295,446 places within hanoi boundary

=== hcmc (Ho Chi Minh City, Vietnam) ===


  downloading Overture places (2026-09-23.1) for bbox (105.9769, 8.3457, 108.4311, 11.5016) ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  saved 538,299 features -> ..\data\overture\hcmc_places_2026-09-23.1.geojson


  dropping 23,451 places with a duplicate name
  403,769 places within hcmc boundary

Cities loaded: ['pleiku', 'can_tho', 'da_nang', 'hanoi', 'hcmc']


---
# 2. Combine cities into one places backup

In [9]:
places_all = gpd.GeoDataFrame(pd.concat(city_places.values(), ignore_index=True), crs="EPSG:4326")
places_all = places_all.dropna(subset=["primary_category"]).reset_index(drop=True)

print(f"Total places across {len(city_places)} cities: {len(places_all):,}")
print(places_all["city"].value_counts().to_string())

places_all.to_file(NEW_BACKUP_PATH, driver="GeoJSON")
print(f"\nBackup saved -> {NEW_BACKUP_PATH}")

Total places across 5 cities: 766,615
city
hcmc       380507
hanoi      273896
da_nang     78041
can_tho     30917
pleiku       3254



Backup saved -> ..\data\overture\places_all_cities_2026-09-23.1.geojson


---
# 3. Category inventory

Every place now carries its own taxonomy path, so the functional group comes straight from
the data: its L1 branch (or L0 group) is mapped by `L1_TO_FUNCTIONAL_GROUP` in
`sitex.arch.poi_classification`. The cached taxonomy sheet is no longer needed for this.

In [10]:
category_counts = (
    places_all.groupby("primary_category")
    .agg(
        count=("primary_category", "size"),
        cities=("city", lambda s: ",".join(sorted(set(s)))),
        hierarchy=("hierarchy", lambda s: s.mode().iloc[0] if s.notna().any() else None),
    )
    .reset_index()
    .rename(columns={"primary_category": "overture_category"})
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
category_counts["functional_group_taxonomy"] = category_counts["hierarchy"].apply(functional_group_from_hierarchy)

print(f"{len(category_counts):,} unique categories in release {RELEASE}")
print(f"{category_counts['functional_group_taxonomy'].isna().sum()} without a functional group from the hierarchy")
category_counts.head(20)

1,161 unique categories in release 2026-09-23.1
0 without a functional group from the hierarchy


,overture_category,count,cities,hierarchy,functional_group_taxonomy
0,restaurant,38145,"can_tho,da_nang,hanoi,hcmc,pleiku",food_and_drink > restaurant,Food & Drink
1,professional_service,33626,"can_tho,da_nang,hanoi,hcmc,pleiku",services_and_business > professional_service,Business & Professional Services
2,coffee_shop,32717,"can_tho,da_nang,hanoi,hcmc,pleiku",food_and_drink > non_alcoholic_beverage_venue ...,Food & Drink
3,shopping,32306,"can_tho,da_nang,hanoi,hcmc,pleiku",shopping,Retail & Shopping
4,real_estate_service,22988,"can_tho,da_nang,hanoi,hcmc,pleiku",services_and_business > real_estate_service,Business & Professional Services
5,spa,21223,"can_tho,da_nang,hanoi,hcmc,pleiku",lifestyle_services > wellness_service > spa,Beauty & Personal Care
6,beauty_salon,16361,"can_tho,da_nang,hanoi,hcmc,pleiku",lifestyle_services > personal_or_beauty_servic...,Beauty & Personal Care
7,cafe,15428,"can_tho,da_nang,hanoi,hcmc,pleiku",food_and_drink > casual_eatery > cafe,Food & Drink
8,clothing_store,14785,"can_tho,da_nang,hanoi,hcmc,pleiku",shopping > fashion_and_apparel_store > clothin...,Retail & Shopping
9,hotel,14656,"can_tho,da_nang,hanoi,hcmc,pleiku",lodging > hotel,Accommodation & Travel


---
# 4. Crosswalk: old category → new category

Two kinds of evidence, strongest first:

- **`id_match`**: the same place (same Overture ID) in the old backup and in the new
  download. Counting the pairs shows where each new category's places came from.
- **`same_name` / `taxonomy_sheet` / `singular`**: for categories with no matched places,
  the old name is found by name: unchanged, renamed in Overture's taxonomy sheet, or
  plural → singular (`printing_services` → `printing_service`).

In [11]:
reviewed = pd.read_csv(REVIEWED_OLD_CSV, encoding="utf-8-sig")
reviewed_cats = set(reviewed["overture_category"])
print(f"Reviewed lookup (old names): {len(reviewed):,} categories")

# --- id_match ---------------------------------------------------------------
old_places = gpd.read_file(OLD_BACKUP_PATH, columns=["id", "primary_category"], ignore_geometry=True)
matched = places_all[["id", "primary_category"]].merge(
    old_places.rename(columns={"primary_category": "old_category"}), on="id", how="inner"
)
id_pairs = (
    matched.groupby(["primary_category", "old_category"]).size().reset_index(name="n_places")
    .rename(columns={"primary_category": "new_category"})
)
id_pairs["method"] = "id_match"
print(f"{len(matched):,} of {len(places_all):,} new places matched to the old backup by ID")

# --- name-based translation of every old category -----------------------------
taxonomy_df = pd.read_csv(TAXONOMY_PATH, encoding="utf-8-sig")
new_names = set(category_counts["overture_category"])
sheet_old_to_new = {}
for row in taxonomy_df.to_dict("records"):
    o, n = str(row.get("Old Primary Category") or "").strip(), str(row.get("New Primary Category") or "").strip()
    if o and n and o != "nan" and n != "nan":
        sheet_old_to_new.setdefault(o, n)


def translate_by_name(old_cat):
    if old_cat in new_names:
        return old_cat, "same_name"
    if sheet_old_to_new.get(old_cat) in new_names:
        return sheet_old_to_new[old_cat], "taxonomy_sheet"
    for cand in ([old_cat[:-3] + "y"] if old_cat.endswith("ies") else []) + ([old_cat[:-1]] if old_cat.endswith("s") else []):
        if cand in new_names:
            return cand, "singular"
    return None, None


name_rows = []
for old_cat, n_old in zip(reviewed["overture_category"], reviewed["count"]):
    new_cat, method = translate_by_name(old_cat)
    if new_cat:
        name_rows.append({"new_category": new_cat, "old_category": old_cat, "n_places": n_old, "method": method})
name_pairs = pd.DataFrame(name_rows)

# Name links only for new categories that have no ID evidence at all.
name_pairs = name_pairs[~name_pairs["new_category"].isin(id_pairs["new_category"])]

crosswalk = pd.concat([id_pairs, name_pairs], ignore_index=True)
crosswalk["old_is_reviewed"] = crosswalk["old_category"].isin(reviewed_cats)
crosswalk = crosswalk.sort_values(["new_category", "n_places"], ascending=[True, False]).reset_index(drop=True)
crosswalk.to_csv(CROSSWALK_CSV, index=False, encoding="utf-8-sig")

print(f"Crosswalk: {len(crosswalk):,} links -> {CROSSWALK_CSV}")
print(crosswalk["method"].value_counts().to_string())

Reviewed lookup (old names): 1,153 categories


699,831 of 766,615 new places matched to the old backup by ID
Crosswalk: 3,994 links -> ..\data\reference\overture_category_crosswalk.csv
method
id_match    3994


---
# 5. Carry the reviewed labels over

For each new category, one reviewed old category is chosen as its **source**:

1. the old category with the **same name**, if there is one (a rename never beats an
   unchanged name);
2. otherwise the old category that most of its places came from (`id_match`);
3. otherwise a name translation (`taxonomy_sheet`, `singular`).

The source's reviewed columns are copied: `functional_group`, `gehl_activity`,
`gehl_activities`, the three Gehl scores, `thirdspace_score` and the review flags.
`gehl_scores` is rebuilt from the three score columns, because those are the columns that
were edited by hand.

`carry_agreement` is the share of the category's ID-matched places whose old category has
the **same labels** as the source. Below 80% (with at least 5 matched places) the row is
flagged: the new category collects places that were reviewed differently before.

In [12]:
REVIEWED_COLS = [
    "functional_group", "functional_group_confidence", "functional_group_source",
    "gehl_activity", "gehl_activities", "necessary_score", "social_score", "optional_score",
    "thirdspace_score", "thirdspace_needs_review", "fg_needs_review", "gehl_needs_review",
]
AGREEMENT_THRESHOLD, AGREEMENT_MIN_PLACES = 0.8, 5

rev = reviewed.set_index("overture_category")
label_key = (rev["functional_group"].astype(str) + "/" + rev["gehl_activities"].astype(str)).to_dict()
links = crosswalk[crosswalk["old_is_reviewed"]]

rows = []
for cat in category_counts["overture_category"]:
    cand = links[links["new_category"] == cat]
    if cat in reviewed_cats:
        source, how = cat, "reviewed_same_name"
    elif len(cand):
        top = cand.iloc[0]          # sorted by n_places, largest first
        source, how = top["old_category"], f"reviewed_{top['method']}"
    else:
        source, how = None, "new"

    ids = cand[cand["method"] == "id_match"]
    n_id = int(ids["n_places"].sum())
    agreement = None
    if source is not None and n_id:
        same = ids["old_category"].map(label_key) == label_key[source]
        agreement = round(ids.loc[same, "n_places"].sum() / n_id, 3)

    rows.append({
        "overture_category": cat, "carry_source": how, "reviewed_from": source,
        "carry_agreement": agreement, "n_id_matched": n_id,
        "other_old_categories": "|".join(o for o in ids["old_category"] if o != source),
    })

carry = pd.DataFrame(rows)
lookup = category_counts.merge(carry, on="overture_category")
lookup = lookup.join(rev[REVIEWED_COLS], on="reviewed_from")

lookup["carry_needs_review"] = (
    (lookup["carry_source"] == "new")
    | ((lookup["carry_agreement"] < AGREEMENT_THRESHOLD) & (lookup["n_id_matched"] >= AGREEMENT_MIN_PLACES))
)

print("── Carry-over ───────────────────────────")
print(lookup["carry_source"].value_counts().to_string())
carried_places = lookup.loc[lookup["carry_source"] != "new", "count"].sum()
print(f"\nPlaces whose category carries reviewed labels: {carried_places:,}/{lookup['count'].sum():,} "
      f"({carried_places / lookup['count'].sum():.1%})")
print(f"Low agreement (flagged): {(lookup['carry_needs_review'] & (lookup['carry_source'] != 'new')).sum()}")

── Carry-over ───────────────────────────
carry_source
reviewed_same_name    805
reviewed_id_match     348
new                     8

Places whose category carries reviewed labels: 766,607/766,615 (100.0%)
Low agreement (flagged): 2


In [13]:
# Reviewed functional group vs. the one the new hierarchy implies -- for information only;
# the reviewed value is kept.
carried = lookup[lookup["carry_source"] != "new"]
differs = carried[carried["functional_group"] != carried["functional_group_taxonomy"]]
print(f"{len(differs)} carried categories where the reviewed functional group differs "
      "from the new hierarchy (reviewed value kept):")
differs[["overture_category", "reviewed_from", "functional_group", "functional_group_taxonomy", "count"]].head(30)

172 carried categories where the reviewed functional group differs from the new hierarchy (reviewed value kept):


,overture_category,reviewed_from,functional_group,functional_group_taxonomy,count
16,auto_dealer,car_dealer,Business & Professional Services,Automotive & Transport,7895
17,historic_site,landmark_and_historical_building,Civic & Government,Culture & Arts,7701
21,building_or_construction_service,construction_services,Business & Professional Services,Home & Garden Services,7254
23,beauty_supply_store,cosmetic_and_beauty_supplies,Beauty & Personal Care,Retail & Shopping,6554
42,automotive_repair,automotive_repair,Business & Professional Services,Automotive & Transport,3791
57,auto_parts_store,auto_parts_and_supply_store,Business & Professional Services,Retail & Shopping,2830
61,warehouse_club_store,wholesale_store,Retail & Shopping,Daily Needs & Grocery,2656
74,karaoke_venue,karaoke,Accommodation & Travel,Culture & Arts,2205
75,motorcycle_repair,motorcycle_repair,Business & Professional Services,Automotive & Transport,2180
83,motorcycle_dealer,motorcycle_dealer,Business & Professional Services,Automotive & Transport,1754


In [14]:
flagged = lookup[lookup["carry_needs_review"] & (lookup["carry_source"] != "new")]
print(f"{len(flagged)} carried categories with low agreement between their old sources:")
flagged[["overture_category", "reviewed_from", "carry_agreement", "n_id_matched",
         "other_old_categories", "functional_group", "gehl_activities", "count"]]

2 carried categories with low agreement between their old sources:


,overture_category,reviewed_from,carry_agreement,n_id_matched,other_old_categories,functional_group,gehl_activities,count
85,sport_league,active_life,0.176,17,pool_billiards|sports_and_recreation_venue|fit...,Recreation & Sport,Social|Necessary|Optional,1704
346,counseling,beauty_salon,0.200,5,financial_advising|health_and_medical|pharmacy...,Beauty & Personal Care,Social,203


---
# 6. BART Gehl activity — only for categories without a reviewed source

Categories marked `new` have no reviewed old category. Their functional group comes from
the hierarchy (above); their Gehl activity is scored here with
`facebook/bart-large-mnli`, one independent score per activity (`multi_label=True`), so a
place can be both Optional and Social. All of them are flagged `needs_review`.

First run downloads the ~1.6&nbsp;GB model; later runs reuse the local HuggingFace cache.

In [15]:
GEHL_LABELS = [
    "necessary activity: compulsory daily task like commuting, healthcare, grocery, or errand",
    "optional activity: recreational or leisure choice like park, museum, sport, or sightseeing",
    "social activity: informal gathering place like cafe, market, playground, or community space",
]
GEHL_LABEL_CLEAN = dict(zip(GEHL_LABELS, ["Necessary", "Optional", "Social"]))
GEHL_MULTI_LABEL_THRESHOLD = 0.50

new_mask = lookup["carry_source"] == "new"
new_cats = lookup.loc[new_mask, "overture_category"].tolist()
print(f"{len(new_cats)} categories need a BART Gehl label")

if new_cats:
    from transformers import pipeline
    classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1)
    results = classifier([c.replace("_", " ") for c in new_cats], GEHL_LABELS,
                         hypothesis_template="This place is a {}.", multi_label=True)
    for cat, res in zip(new_cats, results):
        scores = {GEHL_LABEL_CLEAN[l]: round(s, 3) for l, s in zip(res["labels"], res["scores"])}
        qualifying = [a for a, s in scores.items() if s >= GEHL_MULTI_LABEL_THRESHOLD]
        top = max(scores, key=scores.get)
        i = lookup.index[lookup["overture_category"] == cat][0]
        lookup.loc[i, "gehl_activity"] = top
        lookup.loc[i, "gehl_activities"] = "|".join(qualifying or [top])
        lookup.loc[i, "necessary_score"] = scores["Necessary"]
        lookup.loc[i, "social_score"] = scores["Social"]
        lookup.loc[i, "optional_score"] = scores["Optional"]
        lookup.loc[i, "gehl_needs_review"] = len(qualifying) in (0, 3)

# Functional group for new categories: from the hierarchy.
lookup.loc[new_mask, "functional_group"] = lookup.loc[new_mask, "functional_group_taxonomy"].fillna("Other / Unclassified")
lookup.loc[new_mask, "functional_group_source"] = "taxonomy"
lookup.loc[new_mask, "functional_group_confidence"] = 1.0
lookup.loc[new_mask, "fg_needs_review"] = lookup.loc[new_mask, "functional_group_taxonomy"].isna()
lookup.loc[new_mask, "thirdspace_score"] = lookup.loc[new_mask, "functional_group"].apply(
    lambda fg: 0.0 if fg in THIRDSPACE_EXCLUDED_GROUPS else 1.0
)
lookup.loc[new_mask, "thirdspace_needs_review"] = True

lookup.loc[new_mask, ["overture_category", "count", "hierarchy", "functional_group", "gehl_activities"]].head(30)

8 categories need a BART Gehl label


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

,overture_category,count,hierarchy,functional_group,gehl_activities
1012,basque_restaurant,1,food_and_drink > restaurant > european_restaur...,Food & Drink,Social
1015,video_game_critic,1,services_and_business > media_service > media_...,Business & Professional Services,Social
1025,threading_service,1,lifestyle_services > personal_or_beauty_servic...,Beauty & Personal Care,Social
1060,pathology,1,health_care > outpatient_care_facility > diagn...,Healthcare,Social
1069,ethiopian_restaurant,1,food_and_drink > restaurant > african_restaura...,Food & Drink,Social
1084,industrial_cleaning_service,1,services_and_business > professional_service >...,Business & Professional Services,Necessary
1100,nepalese_restaurant,1,food_and_drink > restaurant > asian_restaurant...,Food & Drink,Social
1123,business_banking_service,1,services_and_business > financial_service > bu...,Business & Professional Services,Necessary


---
# 7. Export the CSV

`gehl_scores` is rebuilt from the three score columns, so hand-edited scores reach the JSON.
`needs_review` is True only for new categories and low-agreement carry-overs. The old
`fg_needs_review` / `gehl_needs_review` columns are kept for reference, but they date from
before the manual review, so they no longer decide what needs review.

**Re-running is safe.** If `overture_place_classification.csv` is already keyed by the new
taxonomy (it has a `carry_source` column), the rows in it are kept exactly as they are and
only categories that are not in it yet are appended, so later hand edits survive.

In [ ]:
lookup["gehl_scores"] = [
    json.dumps({"Necessary": n, "Social": s, "Optional": o})
    for n, s, o in zip(lookup["necessary_score"], lookup["social_score"], lookup["optional_score"])
]
for col in ("fg_needs_review", "gehl_needs_review", "thirdspace_needs_review"):
    lookup[col] = lookup[col].fillna(False).astype(bool)
lookup["needs_review"] = lookup["carry_needs_review"]

EXPORT_COLS = [
    "overture_category", "functional_group", "functional_group_confidence", "functional_group_source",
    "gehl_activity", "gehl_activities", "gehl_scores", "necessary_score", "social_score", "optional_score",
    "thirdspace_score", "thirdspace_needs_review",
    "count", "cities", "fg_needs_review", "gehl_needs_review", "needs_review",
    # new-taxonomy provenance
    "hierarchy", "functional_group_taxonomy", "carry_source", "reviewed_from", "carry_agreement",
    "n_id_matched", "other_old_categories", "carry_needs_review",
]
export_df = lookup[EXPORT_COLS].sort_values(["carry_needs_review", "count"], ascending=[False, False])

existing = pd.read_csv(CLASSIFICATION_CSV, encoding="utf-8-sig") if CLASSIFICATION_CSV.exists() else None
if existing is not None and "carry_source" in existing.columns:
    new_rows = export_df[~export_df["overture_category"].isin(existing["overture_category"])]
    if len(new_rows):
        pd.concat([existing, new_rows], ignore_index=True).to_csv(CLASSIFICATION_CSV, index=False, encoding="utf-8-sig")
        print(f"Appended {len(new_rows):,} new categories; kept {len(existing):,} existing rows untouched.")
    else:
        print(f"No new categories -- {CLASSIFICATION_CSV} left untouched.")
else:
    # The file on disk is the old-taxonomy lookup; its reviewed copy lives in REVIEWED_OLD_CSV.
    export_df.to_csv(CLASSIFICATION_CSV, index=False, encoding="utf-8-sig")
    print(f"Wrote {len(export_df):,} categories (new taxonomy) -> {CLASSIFICATION_CSV}")
    print(f"The reviewed old-taxonomy file stays at {REVIEWED_OLD_CSV}")

print(f"Rows to review: {int(pd.read_csv(CLASSIFICATION_CSV, encoding='utf-8-sig')['needs_review'].sum())}")

---
# 8. Rebuild the JSON from the (hand-edited) CSV

`03-NA02-POIs_Gehl.ipynb` reads **`overture_place_classification.json`**, not the CSV.
The CSV is only where the lookup is reviewed and edited by hand, so an edit reaches the POI
classification only after this cell has turned the CSV into the JSON again.

**After editing the CSV** (in Excel or elsewhere, e.g. the rows with `needs_review = TRUE`):

1. Save the CSV (keep the file name and the UTF-8 encoding).
2. **Restart the kernel**, then run **only this cell** — not the whole notebook (that would
   download the five cities again). The restart makes sure the current `sitex` code is used;
   the cell sets its own paths, so nothing above has to run first.
   It first **checks the edits** (unknown functional group, empty or misspelled Gehl labels —
   `Necessary`, `Optional`, `Social`, pipe-separated —, `thirdspace_score` not 0/1, scores
   outside 0–1, duplicate categories). If it finds a problem it lists the line in the CSV and
   writes nothing; fix it, save, run again. Otherwise it reloads the CSV from disk and writes
   the JSON, keeping the previous JSON as a timestamped backup, and prints how many
   categories are third places.
3. Re-run `03-NA02-POIs_Gehl.ipynb` to write the classified POIs
   (`pois_classified_{slug}.geojson` / `.gpkg`) with the edited labels, then the notebooks
   that use them.

Which columns reach the JSON: `functional_group`, `gehl_activity` and `gehl_activities`
(pipe-separated, e.g. `Optional|Social`), the three Gehl scores and `thirdspace_score`
(1 = third place, 0 = not), all as edited in their own columns. An empty `thirdspace_score`
falls back to the first-pass rule (every functional group except Education, Healthcare and
Civic & Government). The bookkeeping columns
(`count`, `cities`, review flags, provenance) stay in the CSV only.

To count how much was edited compared with the raw automatic output, use
`documentations/00-Data-Overture compare_convert_csv.ipynb`; it also rebuilds the JSON.

In [5]:
import sys
# Self-contained: works right after a kernel restart, without running the cells above.
import json
from pathlib import Path
from sitex.arch.poi_classification import export_classification_json, validate_classification_csv

REF_DIR = (Path("/content/gdrive/MyDrive/Colab_Outputs") if "google.colab" in sys.modules else Path("..") / "data") / "reference"
CLASSIFICATION_CSV = REF_DIR / "overture_place_classification.csv"
CLASSIFICATION_JSON = REF_DIR / "overture_place_classification.json"

# 1. Check the hand edits before writing anything.
problems = validate_classification_csv(CLASSIFICATION_CSV)
if problems:
    print(f"{len(problems)} problem(s) in {CLASSIFICATION_CSV.name} -- fix them in the CSV, save, run this cell again:")
    print(chr(10).join(f"  - {p}" for p in problems))
else:
    # 2. Rebuild the JSON (the previous one is kept as a timestamped backup).
    json_path, backup = export_classification_json(CLASSIFICATION_CSV, CLASSIFICATION_JSON)
    if backup:
        print(f"Backed up existing JSON -> {backup}")
    lookup = json.loads(json_path.read_text(encoding="utf-8"))
    n_third = sum(v["thirdspace_score"] == 1.0 for v in lookup.values())
    print(f"Wrote {len(lookup):,} categories -> {json_path}")
    print(f"  third places: {n_third:,} categories, not third places: {len(lookup) - n_third:,}")

Backed up existing JSON -> ..\data\reference\overture_place_classification_backup_20260930_112324.json
Wrote 1,161 categories -> ..\data\reference\overture_place_classification.json
  third places: 171 categories, not third places: 990
